In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, BooleanType
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
path = f"/Volumes/{catalog_name}/raw/raw_landing/historical-full-load/order_items/landing/"
bronze_checkpoint_path = f"/Volumes/ecommerce/raw/raw_landing/checkpoint/bronze/fact_order_items/"

In [0]:
df = spark.readStream \
.format("delta") \
.table(f"{catalog_name}.bronze.brz_order_items")



In [0]:
# query = df.writeStream \
#     .format("memory") \
#     .queryName("fact_order_items_preview") \
#     .outputMode("append") \
#     .trigger(availableNow=True) \
#     .option("checkpointLocation", f"{bronze_checkpoint_path}_preview") \
#     .start()

# query.awaitTermination()
# display(spark.sql("SELECT * FROM fact_order_items_preview LIMIT 20"))

In [0]:
df = df.dropDuplicates(["order_id","item_seq"])
df = df.withColumn("quantity", F.when(F.col("quantity") == "Two", 2).otherwise(F.col("quantity")).cast("int"))
df = df.withColumn(
    "unit_price",
    F.regexp_replace("unit_price", "[$]", "").cast("double")
)

df = df.withColumn(
    "discount_pct",
    F.regexp_replace("discount_pct", "%", "").cast("double")
)
df = df.withColumn(
    "coupon_code", F.lower(F.trim(F.col("coupon_code")))
)

df = df.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
    .when(F.col("channel") == "app", "Mobile")
    .otherwise(F.col("channel")),
)

df = df.withColumn(
    "processed_time", F.current_timestamp()
)

## save to silver table

In [0]:
silver_checkpoint_path = f"/Volumes/ecommerce/raw/raw_landing/checkpoint/silver/fact_order_items/"

In [0]:
def upsert_to_silver(microBatchDF, batchID):
    table_name = f"{catalog_name}.silver.slv_order_items"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format('delta').mode('overwrite').saveAsTable(table_name)
        spark.sql(
            f'alter table {table_name} set TBLPROPERTIES (delta.enableChangeDataFeed = true)'
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id AND silver_table.item_seq = batch_table.item_seq",
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()  

In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()